In [1]:
# Import packages
import os
import time
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)

# Setup (v0.1)

A notebook that includes functions to run on the full AIS dataset sent across by Windward once it has been put in-place via SFTP. The notebook first documents 'chunking' functions that can be used to break up the ~140GB raw AIS dataset into manageable 'per-IMO' subsets. The notebook then documents a series of functions that can transform each of the 'per-IMO' datasets into a format that will enable it's integration into the EFG module and wider Impact Tracking Architecture.

<u><h5>Input</h5></u>

The input dataset is sourced from Windward and is expected to be around 140 GB in size. The dataset will cover the instantaneous activity of virtually all vessels in the worldwide fleet at an hourly level of temporal resolution. Data fields included in the raw AIS dataset include those associated with the AIS (IMO, ts, lon, lat, sog, cog, reported_draught), fields derived from or associated with AIS (depth, distance_to_shore), oceanic variables (wave height, wave direction, water temperature and salinity, speed and direction of current and swell) and weather (daytime, wind speed and direction, visibility and presence of fog).

<u><h5>1. Chunking</h5></u>

The objective of the 'Chunking' stage is to derive subsets of the raw AIS dataset for each individual IMO number contained in the dataset. From experiences working with UNCTAD's Trade-and-Transport Dataset, modern Macbooks are able to read-in datasets that far exceed the RAM of the individual computer, so long as there is sufficient memory on the laptop's hard-drive. This notebook is therefore designed to read in the full raw AIS dataset, then output this dataset for each IMO number.

<u><h5>2. Filtering for 'Columns of Interest'</h5></u>

For ease of integration into the EFG module and reducing computational load, only data fields useful for computation of the EFG Module are taken forward. These include: imo, ts, lon, lat, sog, cog, distance_to_shore and reported_draught. 

<u><h5>3. Introducing a field for H3 geo-coding</h5></u>

The H3 geocoding system is introduced in order to perform the bulk of the geospatial analysis. The H3 coding system is useful for reducing the dimensionality of datasets. 

<u><h5>4. Introducing 'distance_to_port'</h5></u>

A field representing 'distance_to_shore' is included in the AIS dataset, however 'distance_to_port' is also required for the evaluation of Operational Mode in the EFG Module. An example of a function that evaluates this is included below, applying the Haversine distance function between each point and every port in the port database.

This does however represent a computationally expensive process. If too difficult to execute, it may be best to either: i) amend the 'Operational Mode' algorithm of the EFG Module to one that just uses 'distance_to_shore'; or ii) perhaps derive the 'distance_to_port' field after the Stops/Voyages algorithm has been applied and evaluate 'distance_to_shore' as the minimum distance between the AIS point and the Origin and Destination ports only.

<u><h5>Output</h5></u>

Execution of the function documented in this notebook should then enable access to processed AIS data that can be latterly integrated into the EFG Module, plus provide confidence that this input data is of sufficient quality to provide valid results.

A future version of this notebook may seek to incorporate the following:

<ul>
    <li>Amend the reading and writing functions of the raw AIS data to access and write to UCL Shipping's cloud-based shipping systems, rather than local directories.</li>
    <li>Include H3 codes? Quicker to do calculations on a 'neighbourhood' of H3 codes than it is to perform geospatial analysis, e.g. checking if an H3 code is in a list of H3 codes then rather than if every AIS data point is in that list of H3 codes.</li>
</ul>

## Input

In [3]:
ais_raw = pd.read_csv("/Users/apple/repos/datasets/AIS Data Providers/Windward AI/imo_9120841_2022-07-15.csv", nrows=4)
ais_raw.iloc[:4]

,Unnamed: 0,imo,ts,lon,lat,sog,cog,depth,distance_to_shore,is_daytime,wind_speed,wind_direction,wind_hourly_average_speed,wind_hourly_average_direction,wave_max_height,wave_direction,visibility,fog,water_temperature,salinity,ocean_current_speed,ocean_current_direction,reported_draught,time_to_reported_eta_hours,mean_direction_total_swell,significant_height_total_swell,significant_wave_height_first_swell
0,0,9120841,2022-07-15 01:00:00,-123.310670,-17.271250,17.727340,322.86725,50.0,452.041,1.0,4.565207,81.78036,5.723441,79.586210,7.595774,206.56557,46499.180,0.0,23.885162,36.554960,0.145920,56.715496,11.4,419.5,210.4,3.8,3.47
1,1,9120841,2022-07-15 02:00:00,-123.551030,-17.078350,18.098915,310.90366,50.0,461.247,1.0,5.640740,80.69189,5.472010,81.838554,7.566495,206.97966,51711.863,0.0,23.909750,36.531876,0.189192,45.205235,11.4,418.5,209.6,3.8,3.50
2,2,9120841,2022-07-15 03:00:00,-123.790924,-16.887999,17.418646,294.21225,50.0,470.791,0.0,6.273898,83.40817,6.022469,84.454254,7.561305,207.77396,14862.685,0.0,23.885876,36.535877,0.199763,50.407887,11.4,417.5,209.2,3.9,3.54
3,3,9120841,2022-07-15 04:00:00,-124.082850,-16.812803,17.586258,282.46250,50.0,473.650,0.0,7.674726,70.46073,7.075665,71.728310,7.610441,206.79733,11151.988,0.0,24.004250,36.550545,0.102826,46.593517,11.4,416.5,208.6,3.9,3.52


## 1. Chunking

These functions are designed to first read in the full raw AIS dataset, then output subsets of the raw AIS dataset by IMO number to different CSV files with the naming convention of [IMO number].csv/.

### Locally

In [ ]:
start = time.perf_counter()

# Read-in full Raw AIS data from Local Directory
ais_raw = pd.read_csv("/Users/apple/repos/datasets/AIS Data Providers/Windward AI/imo_9120841_2022-07-15.csv")

finish = time.perf_counter()
print("Finished in {0} minutes.".format(round((finish - start) / 3600, 1)))

In [ ]:
start = time.perf_counter()

# Write 'per-IMO' CSVs to Local Directory
ais_raw_imos = ais_raw.imo.unique()

for imo_idx in ais_raw_imos:

    start_imo = time.perf_counter()

    imo = ais_raw_imos[imo_idx]
    ais_raw[(ais_raw.imo == imo)].to_csv("/Users/apple/repos/datasets/AIS Data Providers/Windward AI/per IMO/{0}.csv".format(imo), index=False)
    
    finish_imo = time.perf_counter()
    print("Processing vessel {0} of {1} (IMO {2}) took {3} seconds.".format(imo_idx, len(ais_raw_imos), imo, round((finish_imo - start_imo), 1)))

finish = time.perf_counter()
print("Finished in {0} minutes.".format(round((finish - start) / 3600, 1)))

### Cloud-based

In [ ]:
# Read-in full Raw AIS data from Cloud-based Directory

In [ ]:
# Write 'per-IMO' CSVs to Cloud-based Directory

## 2. Filtering for 'Columns of Interest'

In [ ]:
start = time.perf_counter()

# Read-in full Raw AIS data from Local Directory
ais_raw = pd.read_csv("/Users/apple/repos/datasets/AIS Data Providers/Windward AI/imo_9120841_2022-07-15.csv")

finish = time.perf_counter()
print("Finished in {0} minutes.".format(round((finish - start) / 3600, 1)))

In [ ]:
# Define columns from the Windward AIS dataset we'd like to take forward
ais_cols = ["imo", "ts", "lon", "lat", "distance_to_shore", "reported_draught", "sog", "cog"]

In [ ]:
# Output Full AIS dataset Filtered for Columns of Interest Only
ais_raw[ais_cols].to_csv("/Users/apple/repos/datasets/AIS Data Providers/Windward AI/per IMO/imo_9120841_2022-07-15 REDUCED.csv".format(imo), index=False)

In [ ]:
# # Output Full AIS dataset Filtered for Columns of Interest Only

## 3. Introducing H3 field

## 4. Evaluating 'distance_to_port' using H3

## Output